# Session 2 — Annotation and measurement

**Question:** What can an LLM output validly measure?

This workbook follows one trace throughout: **research question → input → transformation → raw output → parsed output → validation check**. The comments, reference labels, and cached model output are synthetic course materials. They are designed for stable teaching and are not findings about any named model.

## Learning objectives

By the end, you should be able to:

1. identify the Python type of every major input and output;
2. turn a codebook into an explicit list of messages;
3. inspect cached raw output before parsing it;
4. join model labels to human reference labels;
5. calculate confusion counts, precision, recall, and F1; and
6. explain why apparently high accuracy can still bias a sociological estimate.

## 1. Locate the course files

**Input:** the folder from which the notebook is running.  
**Transformation:** move upward until we find `COURSE_PLAN.md`.  
**Output:** one `Path` object representing the repository root.

Before running the cell, predict the type of `REPO_ROOT`.

In [1]:
from pathlib import Path
import csv
import json
import os
from collections import Counter


def find_repo_root(start):
    """Find the course root without assuming where Jupyter started."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "COURSE_PLAN.md").exists():
            return candidate
    raise FileNotFoundError("Could not find COURSE_PLAN.md")


REPO_ROOT = find_repo_root(Path.cwd())
print(REPO_ROOT)
print(type(REPO_ROOT))

/Users/christopherbarrie/Dropbox/nyu_teaching/GenAI_Soc2026
<class 'pathlib.PosixPath'>


## 2. Load the human-coded input records

The unit of analysis is one synthetic public comment about a rezoning proposal.

**Input:** a CSV file.  
**Transformation:** `csv.DictReader` converts every row into a dictionary; `list(...)` collects the dictionaries.  
**Output:** a list of 12 dictionaries.

Predict the values printed by the three `type(...)` calls.

In [2]:
comments_path = REPO_ROOT / "data" / "session02" / "rezoning_comments.csv"

with comments_path.open(encoding="utf-8", newline="") as file:
    comments = list(csv.DictReader(file))

print("number of comments:", len(comments))
print("outer type:", type(comments))
print("one row type:", type(comments[0]))
print("text value type:", type(comments[0]["text"]))
comments[0]

number of comments: 12
outer type: <class 'list'>
one row type: <class 'dict'>
text value type: <class 'str'>


{'id': '1',
 'speaker_role': 'tenant',
 'text': 'More apartments near the subway would help families like mine stay in the neighborhood, so I support the proposal.',
 'human_label': 'SUPPORT'}

### Stop and interpret

A list represents the collection. Each dictionary represents one case. The column names became dictionary keys. `DictReader` reads the `id` value as a string; Python has not guessed that it represents an integer.

Write down one reason it is useful to inspect the object before transforming it further.

## 3. Represent the codebook explicitly

Our inferential target is the share of speakers who **explicitly support** the proposal. Conditional approval is coded `UNCLEAR`, not `SUPPORT`.

This is the codebook-to-promptbook move in Stuhler, Dang Ton, and Ollion: turn tacit boundaries into objects we can inspect and record.

In [3]:
CODEBOOK = {
    "SUPPORT": "explicit, unqualified endorsement of adopting the proposal",
    "OPPOSE": "explicit rejection or request to vote against the proposal",
    "UNCLEAR": "no stance, a mixed position, or support only if conditions change",
}

valid_labels = list(CODEBOOK.keys())
print(type(CODEBOOK), CODEBOOK)
print(type(valid_labels), valid_labels)

<class 'dict'> {'SUPPORT': 'explicit, unqualified endorsement of adopting the proposal', 'OPPOSE': 'explicit rejection or request to vote against the proposal', 'UNCLEAR': 'no stance, a mixed position, or support only if conditions change'}
<class 'list'> ['SUPPORT', 'OPPOSE', 'UNCLEAR']


## 4. Build the messages for one case

**Input:** one comment dictionary plus stable codebook rules.  
**Transformation:** insert the case text into a user message.  
**Output:** a list containing two message dictionaries.

Before running, predict `messages[1]["role"]` and the type of `messages[1]["content"]`.

In [4]:
SYSTEM_RULES = f"""You are a careful sociological annotator.
Classify stance toward the proposal as one of {valid_labels}.
Rules: {CODEBOOK}
Return JSON with id, label, and verbatim evidence.
Do not infer SUPPORT from favorable issue language when support is conditional."""

comment = comments[2]  # The first difficult conditional case
messages = [
    {"role": "system", "content": SYSTEM_RULES},
    {"role": "user", "content": f"ID: {comment['id']}\nComment: {comment['text']}"},
]

print(messages[1]["role"])
print(type(messages[1]["content"]))
print(messages[1]["content"])

user
<class 'str'>
ID: 3
Comment: I want lower rents, but I cannot support the proposal unless the affordability rules become much stronger.


## 5. Optional live hosted call

The required exercise does **not** need a key. The cell below shows the official OpenRouter client but leaves `RUN_LIVE = False`. If the instructor enables a demonstration, the key is read from the environment and is never printed.

The model identifier is set in one variable so it can be changed without editing every request. Record the actual response metadata when using output for research.

In [5]:
RUN_LIVE = False
COURSE_MODEL = os.getenv("COURSE_MODEL", "openai/gpt-5.2")

if RUN_LIVE:
    from openrouter import OpenRouter

    with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
        response = client.chat.send(
            model=COURSE_MODEL, messages=messages, temperature=0
        )
    live_raw_text = response.choices[0].message.content
    print(live_raw_text)
else:
    print("Live call skipped; continuing with cached teaching output.")

Live call skipped; continuing with cached teaching output.


## 6. Load and inspect cached raw output

**Input:** a JSON file containing metadata and twelve annotations.  
**Transformation:** read the complete file as text first.  
**Output:** one string.

A raw response should be preserved before parsing so later researchers can distinguish what the model returned from what the analysis code derived.

In [6]:
cache_path = REPO_ROOT / "data" / "session02" / "cached_model_output.json"
raw_cache = cache_path.read_text(encoding="utf-8")

print(type(raw_cache))
print(raw_cache[:240] + "…")

<class 'str'>
{
  "record_metadata": {
    "status": "synthetic_cached_demo",
    "created_for": "GenAI in Sociology 2026 Session 2",
    "model": "demo/cached-classifier-v1",
    "provider": "none-live-output-is-instructor-authored",
    "temperature": …


## 7. Parse the string into nested Python objects

`json.loads` changes a string into objects such as dictionaries, lists, strings, numbers, booleans, and `None`. It checks syntax and structure. It does not check whether a label measures the construct correctly.

In [7]:
cached = json.loads(raw_cache)
annotations = cached["annotations"]

print("parsed outer type:", type(cached))
print("annotations type:", type(annotations))
print("one annotation type:", type(annotations[0]))
annotations[2]

parsed outer type: <class 'dict'>
annotations type: <class 'list'>
one annotation type: <class 'dict'>


{'id': 3, 'label': 'SUPPORT', 'evidence': 'I want lower rents'}

## 8. Match model annotations to human reference records

We will make a lookup dictionary from model ID to model annotation. Then we will loop through the human records, copy each dictionary, and add the model fields.

Predict the number of dictionaries in `records` and name the two label keys each will contain.

In [8]:
model_by_id = {str(item["id"]): item for item in annotations}
records = []

for comment in comments:
    model_item = model_by_id[comment["id"]]
    combined = comment.copy()
    combined["model_label"] = model_item["label"]
    combined["model_evidence"] = model_item["evidence"]
    records.append(combined)

print(len(records))
records[2]

12


{'id': '3',
 'speaker_role': 'renter',
 'text': 'I want lower rents, but I cannot support the proposal unless the affordability rules become much stronger.',
 'human_label': 'UNCLEAR',
 'model_label': 'SUPPORT',
 'model_evidence': 'I want lower rents'}

## 9. Start with distributions and simple accuracy

These summaries are useful, but they do not yet show the direction or pattern of error.

In [9]:
human_counts = Counter(row["human_label"] for row in records)
model_counts = Counter(row["model_label"] for row in records)
correct = sum(row["human_label"] == row["model_label"] for row in records)
accuracy = correct / len(records)

print("human labels:", human_counts)
print("model labels:", model_counts)
print("accuracy:", round(accuracy, 3))

human labels: Counter({'SUPPORT': 4, 'OPPOSE': 4, 'UNCLEAR': 4})
model labels: Counter({'SUPPORT': 6, 'OPPOSE': 4, 'UNCLEAR': 2})
accuracy: 0.833


### Interpret before continuing

The model matches 10 of 12 labels, or 83.3%. Yet it produces six support labels where the reference has four. If these labels estimate prevalence, the model changes the answer from 33.3% to 50%.

That is the reading-to-code connection: non-random errors can matter more for the downstream inference than the headline agreement rate.

## 10. Count binary outcomes for SUPPORT

For this check, `SUPPORT` is the positive class. Every other category is not support. The four Boolean branches are mutually exclusive: each record increments exactly one counter.

In [10]:
def count_support_outcomes(records):
    counts = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}

    for record in records:
        human_support = record["human_label"] == "SUPPORT"
        model_support = record["model_label"] == "SUPPORT"

        if human_support and model_support:
            counts["tp"] += 1
        elif not human_support and model_support:
            counts["fp"] += 1
        elif human_support and not model_support:
            counts["fn"] += 1
        else:
            counts["tn"] += 1

    return counts


outcomes = count_support_outcomes(records)
print(outcomes)

{'tp': 4, 'fp': 2, 'fn': 0, 'tn': 6}


## 11. Turn counts into class-specific metrics

- **Precision:** among cases the model calls support, which are support in the reference?
- **Recall:** among reference-support cases, which did the model recover?
- **F1:** the harmonic balance of precision and recall.

`safe_divide` handles a zero denominator explicitly rather than allowing the function to crash.

In [11]:
def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0
    return numerator / denominator


def precision_recall_f1(counts):
    precision = safe_divide(counts["tp"], counts["tp"] + counts["fp"])
    recall = safe_divide(counts["tp"], counts["tp"] + counts["fn"])
    f1 = safe_divide(2 * precision * recall, precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


metrics = precision_recall_f1(outcomes)
print({name: round(value, 3) for name, value in metrics.items()})

{'precision': 0.667, 'recall': 1.0, 'f1': 0.8}


## 12. Inspect the errors as sociological cases

A confusion count says there are two false positives. The records tell us what those errors have in common.

In [12]:
false_positives = [
    row
    for row in records
    if row["human_label"] != "SUPPORT" and row["model_label"] == "SUPPORT"
]

for row in false_positives:
    print("ID:", row["id"], "| role:", row["speaker_role"])
    print("text:", row["text"])
    print("model evidence:", row["model_evidence"])
    print()

ID: 3 | role: renter
text: I want lower rents, but I cannot support the proposal unless the affordability rules become much stronger.
model evidence: I want lower rents

ID: 11 | role: homeowner
text: I could accept additional housing if the city first guarantees a new school and more transit service.
model evidence: I could accept additional housing



### What the errors mean

Both false positives contain favorable language about housing, but both make acceptance conditional. The cached classifier selects the favorable fragment and underweights the clause governing stance.

This is not merely a technical bug. It forces a measurement decision: strengthen the `UNCLEAR` rule, create a `CONDITIONAL` category, or redefine the inferential target as openness under specified conditions.

## 13. Your completion task

Open `assessments/weekly_coding/session02_task.py`. Complete the outcome-counting and metric functions, run its three checks, and submit:

1. your prediction before running the task;
2. the completed functions;
3. passing checks plus an interpretation of the false positives; and
4. a 100–150 word note connecting the error pattern to one required reading.

Weekly work receives completion marks. If one element is missing, you may supply it within one week.

## Optional extension — hosted versus local

Run the same message records through the course OpenRouter model and a local Ollama model. Map both responses into the same record schema. Compare:

- disagreement by case and label;
- support-rate estimate;
- latency and cost;
- model/runtime/version metadata; and
- which result is easier to reproduce.

Neither output is a gold standard. Validate both against the reference and inspect the cases.

## AI-use disclosure template

```text
Tool/model used:
What I used it for:
What I incorporated:
How I checked it:
```

You remain responsible for every claim and every line of code you submit. You should be able to explain this task in the same plain language used in the worked example.